# **🤗 Hugging Face – Creating Embeddingsa**

# 📌 What Are Embeddings?

**Embeddings** are **dense vector representations of text (or images, audio, etc.)** that capture **semantic meaning** in numerical form.

---

## 🔹 Traditional Representation (Sparse)

Text is often represented using sparse vectors like **Bag of Words** or **One-Hot Encoding**:

`"I love AI" → [0, 0, 1, 0, 0, 1, ...]`


❌ Problems:
- Very high dimensional
- No semantic meaning
- “love” and “like” are treated as completely unrelated

---

## 🔹 Embedding Representation (Dense)

Embeddings convert text into **compact, meaningful vectors**:

`"I love AI" → [0.124, -0.882, 0.334, ..., 0.991]`


✔ Typically **256 / 512 / 768+ dimensions**  
✔ Values are **real numbers**

---


Embeddings encode:
- Semantic meaning
- Context
- Relationships between words/sentences


---

## 📐 Similarity Comparison

We compare embeddings using **cosine similarity** or **dot product**:


```
similarity("I love AI", "I enjoy artificial intelligence") → high
similarity("I love AI", "The weather is hot") → low
```

---

## 🚀 Applications of Embeddings

Embeddings power many real-world systems:

- 🔍 **Semantic Search**
- 📚 **Retrieval-Augmented Generation (RAG)**
- 🧩 **Clustering & Topic Modeling**
- 🎯 **Recommendation Systems**
- 💬 **Chatbots & Question Answering**
- 🧠 **Document Similarity & Deduplication**






## **Method 1: Creating Embeddings Using Transformers (AutoModel)**

This method uses a base transformer model like BERT.

**Step 1: Install Dependencies**

In [ ]:
pip install transformers torch

**Step 2: Load Model and Tokenizer**

We’ll use:

- Model: bert-base-uncased

- Library: transformers

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

**Step 3: Tokenize Input Text**

- Text → token IDs

- Adds special tokens ([CLS], [SEP])

- Converts to PyTorch tensors

In [25]:
text = "Hugging Face makes AI development easy."

inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

**Step 4: Generate Embeddings**

In [26]:
with torch.no_grad():
    outputs = model(**inputs)

**The model outputs:**

In [ ]:
outputs.last_hidden_state

**Shape:**

In [ ]:
print(outputs.last_hidden_state.shape)

**Step 5: Convert to Sentence Embedding**

BERT gives token embeddings, not sentence embeddings.

Common approach: **Mean Pooling**

In [ ]:
embeddings = outputs.last_hidden_state
sentence_embedding = embeddings.mean(dim=1)

print(sentence_embedding.shape)

**Step 6: View Actual Embedding Values**

In [ ]:
print(sentence_embedding[0][:10])

## **Method 2 – Using Sentence-Transformers**

This is the best approach for production.

It uses models specifically trained for semantic similarity.

**Install Libraries**

In [ ]:
pip install sentence-transformers

**Code Example**

In [ ]:
from sentence_transformers import SentenceTransformer

# Load pre-trained embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "Hugging Face is amazing.",
    "I love machine learning.",
    "The weather is sunny today."
]

# Generate embeddings
embeddings = model.encode(sentences)

print("Shape:", embeddings.shape)
print("First 10 values of first embedding:")
print(embeddings[0][:10])

### **Comparing Two Sentences (Semantic Similarity)**

Interpretation:

- 0.8+ → Very similar meaning

- 0.5 → Moderate similarity

- 0.1 → Unrelated

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sentence1 = "I love artificial intelligence."
sentence2 = "AI is fascinating."

embeddings = model.encode([sentence1, sentence2])

similarity = cosine_similarity(
    [embeddings[0]],
    [embeddings[1]]
)

print("Cosine Similarity:", similarity[0][0])

### **Simple Semantic Search**

In [ ]:
documents = [
    "Hugging Face builds NLP tools.",
    "Transformers are powerful models.",
    "I enjoy playing football."
]

doc_embeddings = model.encode(documents)

query = "What are transformer models?"
query_embedding = model.encode([query])

similarities = cosine_similarity(query_embedding, doc_embeddings)

print("Similarity scores:", similarities)

### **Batch Processing**

Never encode one sentence at a time in production.

In [ ]:
sentences = [
    "I love machine learning.",
    "Artificial intelligence is fascinating."
]

sentences = ["Sentence " + str(i) for i in range(1000)]

embeddings = model.encode(
    sentences,
    batch_size=32,
    show_progress_bar=True
)

print("Generated:", embeddings.shape)

## **Dimensionality Reduction**

## 🔹 1. PCA Visualization

**PCA (Principal Component Analysis)** is used to **reduce high-dimensional embeddings** (e.g., 768-D) into **2D or 3D** for visualization.

### Why PCA?
- Embeddings are hard to visualize directly
- PCA preserves **maximum variance**
- Helps us **see semantic relationships**

📌 Example Insight:
- Similar sentences appear **closer**
- Different topics form **separate groups**

✅ Common use:
- Debugging embeddings
- Explaining models in demos & presentations

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

sentences = [
    "I love AI",
    "Artificial intelligence is amazing",
    "Football is fun",
    "I enjoy sports"
]

embeddings = model.encode(sentences)

pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

plt.scatter(reduced[:, 0], reduced[:, 1])

for i, txt in enumerate(sentences):
    plt.annotate(txt, (reduced[i, 0], reduced[i, 1]))

plt.show()

## 🔹 2. Clustering Embeddings – KMeans

**KMeans** groups embeddings based on **semantic similarity**.

- Each embedding is a point in vector space
- KMeans assigns points to the nearest cluster center

### What you get:
- Topic-based grouping
- Automatic organization of text

📌 Example Applications:
- Document clustering
- Topic discovery
- News/article grouping
- Dataset exploration

✅ Works best when:
- You don’t know labels beforehand
- You want unsupervised semantic grouping

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=42)
labels = kmeans.fit_predict(embeddings)

for sentence, label in zip(sentences, labels):
    print(f"{sentence} -> Cluster {label}")

## 🔹 3. FAISS – Vector Search (Real Production Level 🚀)

**FAISS (Facebook AI Similarity Search)** is a **high-performance vector database** used for **fast similarity search** on millions/billions of embeddings.

### Why FAISS?
- Extremely fast (C++ backend)
- Scales to large datasets
- Supports approximate & exact search

### FAISS is used for:
- 🔁 **RAG systems**
- 🔍 **Semantic search**
- 📚 **Large-scale retrieval**
- 🧠 **Recommendation engines**

In [ ]:
pip install faiss-cpu

**Basic Vector Search**

In [39]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

query = model.encode(["I like artificial intelligence"])
query = np.array(query)

k = 2
distances, indices = index.search(query, k)

print("Top matches:", indices)
print("Distances:", distances)

Top matches: [[0 1]]
Distances: [[0.380566  0.4637451]]




| **Method** | **When to Use** |
|-----------|----------------|
| **AutoModel (BERT)** | Learning how embeddings work internally, understanding transformer outputs, custom research experiments |
| **SentenceTransformer** | Production systems, RAG pipelines, semantic search, clustering, similarity comparison |

---

### 🔹 Key Insight
- **AutoModel (BERT)** gives **raw token-level embeddings** → more control, more complexity  
- **SentenceTransformer** gives **ready-to-use sentence embeddings** → faster, simpler, optimized for real-world use

✅ **Rule of thumb**:  
- *Learning / Research* → AutoModel  
- *Deployment / Applications* → SentenceTransformer